# Fase 3. Análisis exploratorio de datos

En esta fase describo el conjunto de entrenamiento. Para que el EDA siga siendo fácil de leer, me centro en cinco propiedades fisicoquímicas: peso molecular, LogP, donantes, aceptores y TPSA. No son los únicos descriptores disponibles: calculé el catálogo numérico de RDKit completo y lo usaré para seleccionar predictores en los modelos. Mantengo el conjunto de prueba reservado para evaluarlos.

## 3.1. Lectura de los datos

Leo los CSV preparados en la fase anterior desde la carpeta pública del proyecto. Así puedo ejecutar este notebook sin subir archivos manualmente.

In [ ]:
# Cargo las librerías para leer y analizar las tablas.
from pathlib import Path  # Organizo las rutas locales.
import pandas as pd  # Leo y preparo tablas.
import numpy as np  # Calculo estadísticas numéricas.
from IPython.display import display  # Muestro resultados en Colab.

# Defino una carpeta local para guardar copias de trabajo.
DATA_DIR = Path('/content/datos_dili')  # Uso la carpeta de Colab.
DATA_DIR.mkdir(parents=True, exist_ok=True)  # Creo la carpeta si hace falta.

# Indico la carpeta pública con los datos que preparé en la fase 2.
BASE_URL = 'https://raw.githubusercontent.com/PARODBE/UAX_Fundamento_Ciencia_de_Datos/main/Data/'  # Defino el enlace base.
dilist_original = pd.read_csv(BASE_URL + 'DILIst_FDA.csv')  # Cargo la fuente original.
estructuras = pd.read_csv(BASE_URL + 'estructuras_nombres.csv')  # Cargo el mapa de nombres y estructuras.
datos_limpios = pd.read_csv(BASE_URL + 'dilist_descriptores.csv')  # Cargo los compuestos con todos los descriptores.
train_df = pd.read_csv(BASE_URL + 'dilist_train.csv')  # Cargo entrenamiento.
test_df = pd.read_csv(BASE_URL + 'dilist_test.csv')  # Cargo prueba.

# Creo una tabla completa para revisar cobertura y calidad de estructuras.
dilist = dilist_original.rename(columns={'DILIst_Classification': 'DILI'}).copy()  # Renombro la clase.
datos = dilist.merge(estructuras, on='CompoundName', how='left')  # Uno las etiquetas con el estado de búsqueda.
claves_con_etiqueta = datos.dropna(subset=['InChIKey']).groupby('InChIKey')['DILI'].nunique()  # Cuento clases por estructura.
claves_conflictivas = claves_con_etiqueta[claves_con_etiqueta > 1].index  # Identifico etiquetas contradictorias.
datos_conflicto = datos[datos['InChIKey'].isin(claves_conflictivas)].copy()  # Guardo los registros conflictivos.

# Identifico todos los descriptores numéricos salvo etiquetas e identificadores.
columnas_excluir = ['DILI', 'DILIST_ID', 'InChIKey', 'CID', 'SMILES', 'CompoundName', 'Routes_of_Administration', 'chembl_id']  # Evito datos no moleculares.
DESCRIPTORES = [columna for columna in datos_limpios.select_dtypes(include='number').columns if columna not in columnas_excluir]  # Selecciono propiedades moleculares.

# Guardo copias locales de las tablas de trabajo.
for nombre, tabla in [('dilist_descriptores.csv', datos_limpios), ('dilist_train.csv', train_df), ('dilist_test.csv', test_df)]:  # Recorro las tablas.
    tabla.to_csv(DATA_DIR / nombre, index=False)  # Guardo una copia en Colab.
print('He cargado los datos del repositorio:', DATA_DIR)  # Confirmo la ruta local.
print('Descriptores numéricos disponibles:', len(DESCRIPTORES))  # Muestro el catálogo disponible.

## 3.2. Resumen univariado

Para cada descriptor calculo media, mediana, moda, desviación estándar, mínimo, máximo y cuartiles. También cuento las etiquetas DILI en valores absolutos y relativos.

In [ ]:
# Selecciono cinco propiedades fisicoquímicas fáciles de interpretar.
variables = ['MolWt', 'MolLogP', 'NumHDonors', 'NumHAcceptors', 'TPSA']  # Escojo las variables de Lipinski y TPSA.
variables = [variable for variable in variables if variable in train_df.columns]  # Confirmo que estén en el conjunto.

# Calculo las medidas de resumen univariado con entrenamiento.
resumen = train_df[variables].describe().T  # Obtengo media, desviación, mínimo, cuartiles y máximo.
resumen['mediana'] = train_df[variables].median()  # Añado la mediana.
resumen['moda'] = train_df[variables].mode().iloc[0]  # Añado la moda más frecuente.
display(resumen.round(2))  # Muestro el resumen redondeado.

# Cuento las clases DILI en frecuencia absoluta y relativa.
frecuencias = train_df['DILI'].value_counts().sort_index().to_frame('frecuencia')  # Cuento registros por clase.
frecuencias['porcentaje'] = (100 * frecuencias['frecuencia'] / len(train_df)).round(1)  # Calculo porcentajes.
frecuencias.index = frecuencias.index.map({0: 'DILI negativa', 1: 'DILI positiva'})  # Pongo nombres legibles.
display(frecuencias)  # Muestro la distribución de clases.


In [ ]:
# Importo herramientas para dibujar histogramas y diagramas de caja.
import matplotlib.pyplot as plt  # Creo gráficos.
import seaborn as sns  # Dibujo comparaciones por clase.

# Represento la forma de las distribuciones de cada descriptor.
train_df[variables].hist(figsize=(12, 8), bins=25, edgecolor='black')  # Dibujo un histograma por variable.
plt.suptitle('Distribución de propiedades moleculares en entrenamiento')  # Titulo la figura.
plt.tight_layout()  # Ajusto los espacios entre gráficos.
plt.show()  # Muestro los histogramas.

# Comparo cada propiedad entre las dos clases de DILI.
fig, axes = plt.subplots(1, len(variables), figsize=(4 * len(variables), 4))  # Creo un panel de diagramas.
for ax, variable in zip(axes, variables):  # Recorro cada descriptor y su eje.
    sns.boxplot(data=train_df, x='DILI', y=variable, ax=ax)  # Dibujo cajas por clase.
    ax.set_title(variable)  # Pongo el nombre del descriptor.
plt.tight_layout()  # Acomodo los títulos.
plt.show()  # Muestro las comparaciones.


## 3.3. Calidad de los datos y valores atípicos

Reviso nulos, duplicados y posibles valores atípicos. Para los atípicos uso el criterio exploratorio de 1,5 veces el rango intercuartílico. No los elimino automáticamente porque pueden corresponder a moléculas reales.

In [ ]:
# Calculo la cantidad y el porcentaje de valores ausentes en la tabla completa.
calidad = pd.DataFrame({'nulos': datos_limpios.isna().sum(), 'porcentaje_nulos': (100 * datos_limpios.isna().mean()).round(1)})  # Resumo los nulos de la tabla molecular depurada.
display(calidad.sort_values('porcentaje_nulos', ascending=False).head(12))  # Muestro primero las columnas con más nulos.
print('Nombres sin estructura inequívoca:', datos['InChIKey'].isna().sum())  # Informo la cobertura de estructuras en la fuente.

# Cuento registros y estructuras duplicadas en la tabla de entrenamiento.
print('Nombres repetidos en DILIst:', dilist['CompoundName'].duplicated().sum())  # Cuento nombres repetidos.
print('InChIKey repetidas en train:', train_df['InChIKey'].duplicated().sum())  # Compruebo duplicados químicos.
print('Filas con etiquetas contradictorias:', len(datos_conflicto))  # Muestro los conflictos detectados.

# Cuento posibles valores atípicos por descriptor usando el rango intercuartílico.
outliers = []  # Creo una lista para guardar los recuentos.
for variable in variables:  # Reviso cada propiedad seleccionada.
    q1 = train_df[variable].quantile(0.25)  # Calculo el primer cuartil.
    q3 = train_df[variable].quantile(0.75)  # Calculo el tercer cuartil.
    iqr = q3 - q1  # Calculo el rango intercuartílico.
    cantidad = ((train_df[variable] < q1 - 1.5 * iqr) | (train_df[variable] > q3 + 1.5 * iqr)).sum()  # Cuento valores fuera de los límites.
    outliers.append({'variable': variable, 'cantidad': cantidad, 'porcentaje': round(100 * cantidad / len(train_df), 1)})  # Guardo el resultado.
display(pd.DataFrame(outliers))  # Muestro los recuentos de posibles atípicos.


## 3.4. Análisis bivariado y multivariado

Uso la correlación de Spearman porque las relaciones pueden no ser lineales y algunos descriptores son discretos. También comparo las medianas y la correlación de cada propiedad con la etiqueta DILI.

In [ ]:
# Calculo la correlación entre las cinco propiedades fisicoquímicas.
correlaciones = train_df[variables].corr(method='spearman')  # Calculo Spearman con entrenamiento.
sns.heatmap(correlaciones, annot=True, fmt='.2f', cmap='coolwarm', center=0)  # Represento la matriz.
plt.title('Correlación de Spearman entre propiedades')  # Titulo el mapa.
plt.tight_layout()  # Ajusto la figura.
plt.show()  # Muestro la correlación.

# Comparo la mediana de las propiedades en las clases DILI.
medianas = train_df.groupby('DILI')[variables].median().T  # Calculo las medianas por clase.
medianas['diferencia positiva-negativa'] = medianas[1] - medianas[0]  # Calculo una diferencia descriptiva.
display(medianas.sort_values('diferencia positiva-negativa', key=abs, ascending=False).round(2))  # Ordeno por diferencia absoluta.

# Calculo la asociación monotónica de cada variable con DILI.
relacion = train_df[variables + ['DILI']].corr(method='spearman')['DILI'].drop('DILI')  # Extraigo la correlación con la etiqueta.
display(relacion.sort_values(key=abs, ascending=False).to_frame('Spearman con DILI'))  # Muestro las asociaciones.


## ¿Qué me dicen los datos?

Interpreto los resultados observando el balance de las clases, las distribuciones y las diferencias entre medianas. La correlación describe asociación, no demuestra que una propiedad cause hepatotoxicidad. Las conclusiones deben limitarse a los compuestos incluidos en DILIst y con estructura identificada.

In [ ]:
# Resumo el balance observado en el conjunto de entrenamiento.
proporciones = train_df['DILI'].value_counts(normalize=True)  # Calculo las proporciones por clase.
print(f'En train, la etiqueta positiva representa {100 * proporciones.get(1, 0):.1f}% y la negativa {100 * proporciones.get(0, 0):.1f}%.')  # Explico el balance.

# Resumo las medianas de MolLogP por clase como lectura descriptiva.
medianas_logp = train_df.groupby('DILI')['MolLogP'].median()  # Calculo medianas de LogP por etiqueta.
print(f'La mediana de MolLogP es {medianas_logp.get(1, np.nan):.2f} en DILI positiva y {medianas_logp.get(0, np.nan):.2f} en DILI negativa.')  # Muestro ambas medianas.

# Indico que los valores extremos quedan para revisión y no se borran automáticamente.
print('Mantengo los posibles valores atípicos para revisarlos en contexto químico, en vez de eliminarlos automáticamente.')  # Explico mi decisión.
